<a href="https://colab.research.google.com/github/kevin-blasiak-curtin/ISYS2001-Archive/blob/main/Module%2007%20-%20Directing%20Pandas/pandas_lecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas: business data processing in Python

Pandas is Python's main tool for analysing data: taking a table of raw numbers and
pulling business insight out of it, such as totals, averages, and comparisons.

Today:

- Where pandas came from, and what it is
- Getting data into it, and looking at it first
- Cleaning, grouping, and analysing for insight
- How it connects to last week's dictionaries

**Following along?** Open the module folder here:
https://github.com/kevin-blasiak-curtin/ISYS2001-Archive/tree/main/Module%2007%20-%20Directing%20Pandas

---
## Part 1: What pandas is, and where it came from

- 2008: Wes McKinney, working at an investment firm, needed to pull business insight out of financial data in Python
- Python had no good tool for it at the time; the usual choice was another language, R
- He needed to stay in Python, so he built the missing piece himself and released it as pandas
- In effect, he was automating a chunk of his own analysis work
- The name is from "panel data" (an economics term for data tracked over time), and a pun on "Python data analysis"
- Not, as you might guess, from the `df` we are about to use

### In a nutshell

- Last week: dictionaries, lists, and lists of lists, all built by hand
- Pandas is essentially that, packaged up as a library
- Someone has already done the fiddly work, so you do not rebuild it every time

Here is the kind of thing we did last week, as a quick refresher:

In [ ]:
# Last week: a dictionary holds named values
prices = {'chair': 299.00, 'lamp': 45.50, 'monitor': 199.00}

# We can look one up, or add them all up
print(prices['lamp'])
print(sum(prices.values()))

### The two things pandas gives you

- **Series**: a single labelled column (think: one labelled list)
- **DataFrame**: the whole table, named columns with rows lined up (think: a spreadsheet in one variable)
- A DataFrame is really just several Series sharing the same rows
- Almost everything in pandas is one of these two

---
## Part 2: A few practical things to know

**Installing it**

- Pandas sometimes has to be installed before you can use it
- Colab: Google has already installed it, so we just tell Python we want it, with the line below
- Your own machine (VS Code, say): install it once from the terminal first, with `pip install pandas`
- Run this next cell now; if a later cell says `pd is not defined`, it means this one has not been run

In [ ]:
import pandas as pd

**Where the data comes from**

- Different from last week: the data is not typed into the code, it arrives in a file
- Usually a CSV (comma separated values); also Excel, JSON, or a database

**Where the data goes: into RAM**

- Loading a file reads it into RAM, the computer's working memory
- Like human working memory, it clears when the session ends, so closing Colab means re-uploading
- Real limitation: it all has to fit in memory, so very large files are a problem
- Our practice files are tiny, so this never bothers us

### Getting the file into RAM

- I have downloaded a small sample file for us to use
- Good practice: look at it first, in a text editor or Excel, before loading it
- We can see it is a CSV, a few columns, some sales data
- Now upload it to Colab: drag it into the file panel on the left (folder icon)
- Session times out? The file is gone, so upload it again
- On your own machine you would not upload; the file already sits in your project folder
- Let us also read it here, to see what we are dealing with

In [ ]:
with open('sales_data.csv') as f:
    for line in f:
        print(line.strip())

- Sales data: product, quantity, unit price, total, rep, region
- Several columns carry dollar signs, so they are text, not numbers yet
- Trap: `Quantity` has dollar signs too, but it is a count of units, not money
- Strip every sign blindly and 15 chairs becomes $15
- This is exactly why we look before we clean

---
## Part 3: Load it, clean it, put it to work

- Sometimes data arrives in a form we cannot use directly, like those dollar signs
- Pandas helps us load it and get it ready
- `read_csv` loads the whole file in one line; `head` shows the first rows to confirm

In [ ]:
df = pd.read_csv('sales_data.csv')
df.head()

- `info`: how many rows, the columns, and each column's type
- Notice the money columns arrive as text (`object`) because of the dollar signs

In [ ]:
df.info()

**Cleaning `Total_Sale`**

- It is text like `$4998.00`
- Strip the `$`, then convert the column to numbers
- Read the two lines: clean, then convert. Done once for the whole column, not row by row

In [ ]:
df['Total_Sale'] = df['Total_Sale'].str.replace('$', '', regex=False)
df['Total_Sale'] = pd.to_numeric(df['Total_Sale'])
df['Total_Sale']

**Put pandas to work: group and summarise**

- The most common move: group rows together, then summarise each group
- Here: group the sales by region, and total the sale value in each. One line

In [ ]:
df.groupby('Region')['Total_Sale'].sum()

**Pandas can do more: total, then sort**

- Chain a sort onto the total, so the strongest region comes first
- Watch the table reorder

In [ ]:
df.groupby('Region')['Total_Sale'].sum().sort_values(ascending=False)

**A few other everyday operations, so you know they exist**

In [ ]:
df['Total_Sale'].mean()               # the average sale value

In [ ]:
df['Region'].value_counts()           # how many sales in each region

In [ ]:
df[df['Region'] == 'NSW']             # just the rows for one region

**Pandas can do far more than this. To find out what:**

- Ask an AI to list the options for your problem, then check them against what you actually want
- Or the old-school way, the pandas docs. Missing data, for instance, is a very common real problem:
  https://pandas.pydata.org/docs/user_guide/missing_data.html

### Stretch goal: a quick chart

- Pandas can turn a result straight into a chart, by handing off to another library, matplotlib
- A taste of where this goes next, not something we go deep on today
- Take the region totals and draw them as a bar chart in a couple of lines

In [ ]:
import matplotlib.pyplot as plt

region_totals = df.groupby('Region')['Total_Sale'].sum().sort_values(ascending=False)
region_totals.plot(kind='bar', title='Total sales by region')
plt.ylabel('Total sales ($)')
plt.tight_layout()
plt.show()

---
## Part 4: Isn't this just last week's dictionaries?

- Short answer: pretty much, yes
- You could recreate all of this with the dictionaries and lists you already know
- But that is a lot of hard work, and McKinney has already done it for us in the library
- Same job, sales by region, done both ways below

**The dictionary way**

- Read each row, look up the region, keep a running total, sort at the end
- Note the if/else needed to build the totals up by hand

In [ ]:
import csv

sales_by_region = {}

with open('sales_data.csv') as f:
    for row in csv.DictReader(f):
        region = row['Region']
        amount = float(row['Total_Sale'].replace('$', ''))
        if region in sales_by_region:
            sales_by_region[region] += amount
        else:
            sales_by_region[region] = amount

for region, total in sorted(sales_by_region.items(), key=lambda pair: pair[1], reverse=True):
    print(f"{region:<6} {total:>10.2f}")

**The same result in pandas**

In [ ]:
df.groupby('Region')['Total_Sale'].sum().sort_values(ascending=False)

**The difference**

- Dictionary: a loop plus if/else, built by hand
- Pandas: one line, because the grouping is already built in
- Less to write, and less to get wrong

**So why keep dictionaries at all?**

- Still the right tool for small jobs: a few values, a single lookup, a one-off
- Fine control, and quick to throw together
- But for a real table of data, most of business, pandas makes the job far easier

---
## A note on limits

- The data must fit in RAM: fine for us, a constraint for very large or ML-scale data
- Workarounds exist (chunking the file, or libraries like Polars or Dask), beyond our scope
- If you ever hit that wall, you will know what to search for

## What we covered

- What pandas is, and why McKinney built it
- Series and DataFrame
- Getting a file into memory, and looking first
- Clean, group, sort for insight
- A first chart, as a taste of what is next
- Dictionaries versus pandas, and when each one fits